# COT Positioning Dashboard — companion analysis

Cross-market positioning-extremes from the CFTC Commitment of Traders report (Legacy, Futures-Only, dataset `6dca-aqww`).

**Metrics (causal, trailing window only):**
- net position = long − short (spec / commercial)
- OI normalization = net ÷ open interest
- rolling z-score & percentile, 156-week window, min 52 weeks
- extreme: pct ≥ 90 → Extreme Long · pct ≤ 10 → Extreme Short
- confluence: spec & commercial extremes in opposing directions
- week-over-week change in net% (percentage points)

> Descriptive only — not a trading signal. Percentiles are relative to each market's *own* trailing 3-year range.

In [ ]:
# Imports + shared registry
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

from market_registry import MARKETS, SECTOR_ORDER
from metrics import compute_metrics, latest_row
from sample import build_sample_data

In [ ]:
# Load data — synthetic sample by default. For live data:
#   from fetch import fetch_market_frames; frames = fetch_market_frames(years=4)
frames = build_sample_data()
print(f"{len(frames)} markets loaded")

In [ ]:
# Compute metrics for every market, keep the latest row
summary = []
for meta in MARKETS:
    df = frames.get(meta["code"])
    if df is None:
        continue
    row = latest_row(compute_metrics(df))
    if row is None:
        continue
    summary.append({
        "ticker": meta["ticker"], "name": meta["name"], "sector": meta["sector"],
        "date": row["date"],
        "spec_pct": round(float(row["spec_pct"]), 1),
        "comm_pct": round(float(row["comm_pct"]), 1),
        "spec_z": round(float(row["spec_z"]), 2),
        "comm_z": round(float(row["comm_z"]), 2),
        "spec_extreme": row["spec_extreme"],
        "comm_extreme": row["comm_extreme"],
        "confluence": bool(row["confluence"]),
        "spec_wow": (None if pd.isna(row["spec_wow"]) else round(float(row["spec_wow"]), 2)),
    })

df_sum = pd.DataFrame(summary)
df_sum = df_sum.sort_values("spec_pct", ascending=False)
df_sum

In [ ]:
# Ranked by extremity and confluence flags
df_sum["extremity"] = (df_sum["spec_pct"] - 50).abs()
df_sum.sort_values("extremity", ascending=False).head(12)

In [ ]:
# Per-market series — inspect one market's full history
code = df_sum.iloc[0]["code"]
meta = next(m for m in MARKETS if m["code"] == code)
print(meta["ticker"], meta["name"])
compute_metrics(frames[code]).tail(8)

In [ ]:
# Heatmap: sectors × markets, colored by latest spec percentile
from heatmap import build_matrix, plot
import matplotlib.pyplot as plt

data, confluent, sectors, tickers, _ = build_matrix()
fig = plot(data, confluent, sectors, tickers)
plt.show()

## Notes & caveats

- No price/momentum filter — a market can stay at an extreme through a long trend.
- Legacy report's commercial category in commodities blends hedgers with swap dealers; Disaggregated would be sharper for Ag/Energy/Metals.
- Positioning extremes and reversals have no validated historical relationship here — descriptive, not predictive.
- Verify `market_registry.py` codes against the current CFTC report periodically.